# Exploratory Data Analysis of OBD-II Time Series

Goal:
To analyze sensor distributions, temporal behavior and correlations
in OBD-II driving data before feature engineering and modeling.

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

In [ ]:
plt.style.use("seaborn-v0_8")

In [ ]:
data_path = Path("../data/processed/clean_obd_data.csv")

df = pd.read_csv(data_path)

df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df.describe()

Distribution Analaysis

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df["speed"], bins=60)
plt.title("Speed Distribution")
plt.xlabel("Speed")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df["rpm"], bins=60)
plt.title("RPM Distribution")
plt.xlabel("RPM")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df["throttle"], bins=60)
plt.title("Throttle Distribution")
plt.xlabel("Throttle % ")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df["pedal_d"], bins=60)
plt.title("Accelerator Pedal Position Distribution")
plt.xlabel("Pedal Position %")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df["map"], bins=60)
plt.title("MAP Distribution")
plt.xlabel("MAP")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df["maf"], bins=60)
plt.title("MAF Distribution")
plt.xlabel("MAF")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df["coolant_temp"], bins=60)
plt.title("Coolant Temperature Distribution")
plt.xlabel("Coolant Temperature")
plt.ylabel("Frequency")
plt.show()

Time Series Inspection

In [ ]:
trip_id = df["trip_id"].unique()[0]

sample_trip = df[df["trip_id"] == trip_id]

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(sample_trip["speed"])
plt.title("Speed Time Series")
plt.xlabel("Time Step")
plt.ylabel("Speed")
plt.show()

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(sample_trip["rpm"])
plt.title("RPM Time Series")
plt.xlabel("Time Step")
plt.ylabel("RPM")
plt.show()

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(sample_trip["throttle"])
plt.title("Throttle Time Series")
plt.xlabel("Time Step")
plt.ylabel("Throttle")
plt.show()

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(sample_trip["map"])
plt.title("MAP Time Series")
plt.xlabel("Time Step")
plt.ylabel("MAP")
plt.show()

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(sample_trip["coolant_temp"])
plt.title("Coolant Temperature Time Series")
plt.xlabel("Time Step")
plt.ylabel("Coolant Temp")
plt.show()

Sensor Relationship Analysis

In [ ]:
plt.figure(figsize=(6,5))
sns.scatterplot(x=df["throttle"], y=df["rpm"], alpha=0.3)
plt.title("Throttle vs RPM")
plt.xlabel("Throttle")
plt.ylabel("RPM")
plt.show()

In [ ]:
plt.figure(figsize=(6,5))
sns.scatterplot(x=df["rpm"], y=df["speed"], alpha=0.3)
plt.title("RPM vs Speed")
plt.xlabel("RPM")
plt.ylabel("Speed")
plt.show()

In [ ]:
plt.figure(figsize=(6,5))
sns.scatterplot(x=df["map"], y=df["rpm"], alpha=0.3)
plt.title("MAP vs RPM")
plt.xlabel("MAP")
plt.ylabel("RPM")
plt.show()

Correlation Matrix

In [ ]:
corr_matrix = df.corr(numeric_only=True)

In [ ]:
plt.figure(figsize=(12,10))
sns.heatmap(corr_matrix, cmap="coolwarm", center=0)
plt.title("Sensor Correlation Matrix")
plt.show()

Trip Level Analysis

In [ ]:
trip_statistics = df.groupby("trip_id").agg(
{
    "speed":["mean","max"],
    "rpm":["mean","max"],
    "throttle":"mean",
    "map":"mean",
    "maf":"mean"
}
)

trip_statistics

Outliers Detection

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x=df["speed"])
plt.title("Speed Outliers")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x=df["rpm"])
plt.title("RPM Outliers")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x=df["throttle"])
plt.title("Throttle Outliers")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x=df["map"])
plt.title("MAP Outliers")
plt.show()

In [ ]:
df.var(numeric_only=True).sort_values()

## Observations from Exploratory Data Analysis

After exploring the OBD-II sensor data through distributions, time-series plots, scatter plots, and correlation analysis, several useful patterns became visible in the dataset.

The dataset contains multiple trips with continuous measurements of vehicle behaviour such as speed, engine RPM, throttle position, accelerator pedal position (Pedal D), manifold absolute pressure (MAP), mass air flow (MAF), and coolant temperature. From the distribution plots, most values fall within reasonable operating ranges, which indicates that the data was recorded under normal driving conditions and does not contain obvious sensor failures.

Looking at the time-series plots for individual trips, the signals behave in a way that matches real driving behaviour. Speed and RPM change frequently depending on acceleration and braking, while throttle and pedal position fluctuate as the driver presses or releases the accelerator. Coolant temperature changes more gradually, which is expected since engine temperature usually increases slowly as the engine warms up during a trip.

The scatter plots also reveal clear relationships between some variables. RPM tends to increase when vehicle speed increases, which reflects normal engine behaviour. Similarly, higher throttle values are often associated with higher RPM, indicating that the engine responds to the driver’s acceleration input. MAP values show higher readings when engine load increases, especially during periods of acceleration.

The correlation matrix further supports these observations by highlighting stronger relationships between variables such as speed, RPM, throttle, and pedal position. These signals are closely related because they represent different parts of the driver-vehicle interaction process. In contrast, variables like coolant temperature show weaker correlations with the other signals since they represent thermal conditions rather than direct driving inputs.

Trip-level statistics show that different trips have different average speeds and engine activity levels, suggesting that the dataset includes a mixture of driving conditions such as slower urban driving and more consistent cruising periods.

Boxplots also reveal a few extreme values in speed, RPM, and throttle. These points may correspond to aggressive acceleration events or short bursts of high engine activity, which could be important for later behaviour analysis.

Overall, the dataset appears consistent and realistic, with meaningful variation in the main driving signals. This indicates that the data is suitable for the next stages of the project, including preprocessing, feature engineering, and modelling driver behaviour patterns.